[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [27]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.head_dim = d_model // num_heads
        self.num_heads = num_heads

        self.Wqs = nn.ModuleList(
            [nn.Linear(d_model, self.head_dim) for _ in  range(num_heads)]
        )
        self.Wks = nn.ModuleList(
            [nn.Linear(d_model, self.head_dim) for _ in  range(num_heads)]
        )
        self.Wvs = nn.ModuleList(
            [nn.Linear(d_model, self.head_dim) for _ in  range(num_heads)]
        )

        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
      """
      x_q: (B, S_q, d_model) from DECODER
      x_kv: (B, S_kv, d_model) from ENCODER

      return: (B, S_q, d_model)
      """

      Q = torch.stack([Wq(x_q) for Wq in self.Wqs], dim = 0) #(num_heads, B, S_q, head_dim)
      K = torch.stack([Wk(x_kv) for Wk in self.Wks], dim = 0) #(num_heads, B, S_kv, head_dim)
      V = torch.stack([Wv(x_kv) for Wv in self.Wvs], dim = 0) #(num_heads, B, S_kv, head_dim)

      d_k = K.shape[-1]

      att_weights = nn.functional.softmax(((Q @ K.permute(0, 1, 3, 2)) / (d_k ** 0.5)), dim = -1) #(num_heads, B, S_q, S_kv)

      out = att_weights @ V #(num_heads, B, S_q, head_dim)

      B = out.shape[1]
      S_q = out.shape[2]
      out = out.permute(1, 2, 0, 3).reshape(B, S_q, self.d_model)#(B, S_q, head_dim * num_heads = d_model)

      return self.Wo(out)


In [28]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [31]:
# ✅ SUBMIT
from torch_judge import check, status
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (4.9ms)
  ✅ [2/4] Q and KV different lengths (1.0ms)
  ✅ [3/4] No causal mask — all KV affects all Q (2.9ms)
  ✅ [4/4] Gradient flow (1.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (10.5ms total)
  Progress saved. Run status() to see your dashboard.

